# 🕵️ Your First Anonymization

Detect sensitive entities and replace them with LLM-generated substitutes --
the simplest end-to-end example of Anonymizer.

#### 📚 What you'll learn

- Load a CSV dataset and configure Anonymizer in a few lines
- Preview anonymized results on a small sample before committing to a full run
- Inspect entity detection and replacement with `display_record()`
- Process the full dataset with `run()`

> **Tip:** First time running notebooks? Start with
> [setup instructions](https://nvidia-nemo.github.io/Anonymizer/latest/tutorials/).

## ⚙️ Setup

- Check if your `NVIDIA_API_KEY` from [build.nvidia.com](https://build.nvidia.com) is registered for model access.
- Import the core Anonymizer classes: `Anonymizer`, `AnonymizerConfig`, `AnonymizerInput`, and `Substitute`.
- `Anonymizer()` initializes with the default model provider -- no extra config needed.
- `Anonymizer.configure_logging()` controls verbosity -- switch to `Anonymizer.configure_logging(LoggingConfig.debug())` when troubleshooting.

In [7]:
import getpass
import os

if not os.getenv("NVIDIA_API_KEY"):
    key = getpass.getpass("Enter NVIDIA_API_KEY from build.nvidia.com: ").strip()
    if not key:
        raise RuntimeError("NVIDIA_API_KEY is required to run these notebooks.")
os.environ["NVIDIA_API_KEY"] = "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"
    

In [8]:
from anonymizer import Anonymizer, AnonymizerConfig, AnonymizerInput, Substitute

In [9]:
anonymizer = Anonymizer()

[10:48:25] [INFO] 🔧 Anonymizer initialized with 3 model configs
[10:48:25] [INFO]   |-- 🔎 detector:  gliner-pii-detector
[10:48:25] [INFO]   |-- ✅ validator: gpt-oss-120b
[10:48:25] [INFO]   |-- 🧩 augmenter: gpt-oss-120b


## 📦 Load data and configure

- `AnonymizerInput` points to your CSV and names the text column. `data_summary`
  gives the LLM context about the kind of text it will process.
- Records up to 2,000 tokens each work with the default model configs.
- `AnonymizerConfig` with `Substitute()` tells Anonymizer to replace detected
  entities with LLM-generated synthetic values for names, cities, dates, etc.

In [10]:
input_data = AnonymizerInput(
    source="https://raw.githubusercontent.com/NVIDIA-NeMo/Anonymizer/refs/heads/main/docs/data/NVIDIA_synthetic_biographies.csv",
    text_column="biography",
    data_summary="Biographical profiles of individuals",
)

config = AnonymizerConfig(replace=Substitute())

## 👁️ Preview

- `preview()` runs on a small sample so you can iterate quickly.
- Always preview before processing the full dataset -- it's the fastest way
  to catch prompt or config issues early.

In [11]:
preview = anonymizer.preview(config=config, data=input_data, num_records=3)

[10:48:27] [INFO] 📂 Loaded 25 records from https://raw.githubusercontent.com/NVIDIA-NeMo/Anonymizer/refs/heads/main/docs/data/NVIDIA_synthetic_biographies.csv (column: 'biography')
[10:48:27] [INFO] detection labels in scope: (default: 65 labels; see anonymizer.DEFAULT_ENTITY_LABELS for list)
[10:48:27] [INFO]   |-- 👀 Preview mode: processing 3 of 25 records
[10:48:27] [INFO] 🔍 Running entity detection on 3 records
[10:48:27] [WARNING] Replacement keys not found in template: ['<<TAG_NOTATION>>']
[10:49:31] [INFO]   |-- 📋 Detection complete — 79 entities found across 3 records (0 failed) [63.6s]
[10:49:31] [INFO]   |-- labels: first_name=22, organization_name=8, age=5, occupation=5, city=4, state=4, degree=4, university=4, field_of_study=4, last_name=3, race_ethnicity=3, political_view=3, language=2, religious_belief=2, street_address=2, place_name=1, date_of_birth=1, employment_status=1, company_name=1
[10:49:31] [INFO] 🔄 Running Substitute replacement
[10:49:58] [WARNING] ⚠️ Generatio

## 🔍 Inspect

- `display_record()` shows the original text with highlighted entities,
  the replacement map, and the anonymized output -- all in one view.
- The result dataframe has original and substituted text side-by-side.

In [ ]:
preview.display_record(0)

In [ ]:
preview.display_record(1)

In [ ]:
preview.dataframe

## 🚀 Full run

- `run()` processes the entire dataset with the same config you previewed.
- Access the output via `result.dataframe`.

In [ ]:
result = anonymizer.run(config=config, data=input_data)
print(result)

In [ ]:
result.dataframe.head()

## ⏭️ Next steps

- **[🔍 Inspecting Detected Entities](02_inspecting_detected_entities.ipynb)** --
  dig into what the detection pipeline found and debug quality.
- **[🎯 Choosing a Replacement Strategy](03_choosing_a_replacement_strategy.ipynb)** --
  compare Redact, Annotate, Hash, and Substitute side-by-side.
- **[✏️ Rewriting Biographies](04_rewriting_biographies.ipynb)** --
  generate privacy-safe paraphrases instead of token-level replacements.